# Basic Coalescent Model


## Simpel Markov-model
En Markov jump proces bevæger sig mellem forskellige states over tid. 
Hver state beskrives som en vektor af heltal, der repræsenterer systemets egenskaber.

Overgange mellem states bestemmes af en funktion, som for en given state returnerer:

* hvilke nye states der kan nås
* med hvilke rater (hastigheder)

Denne funktion kaldes en callback-funktion og er central i opbygningen af modellen.

Denne model svarer til en eksponentiel fordeling, hvor processen kun har et trin før den afsluttes.

In [ ]:
import phasic 
from phasic import Graph # ALWAYS import phasic first to set jax backend correctly
import numpy as np


def model(state):
    transitions = []               # List for transitions from state
    if state[0] < 2:               # Control transitions allowed
        child = state.copy()       # Make a copy to modify
        child[0] += 1              # Make the child state
        trans = [child, 2.0]       # Pair the child with a transition rate (in a list not tuple)
        transitions.append(trans)  # Append transition 
    return transitions

graph = Graph(model, ipv=[1])
graph.plot()

: 

## Analyse af modellen 

In [ ]:
graph.expectation(), graph.variance()

Forventet ventetid $ = \frac{1}{rate} = 0.5$

## Momenter

In [ ]:
graph.moments(10)

Expectation: 1.5
Variance: 1.1388888888888884
Moments: [1.5, 3.3888888888888884, 10.58333333333333, 42.9074074074074, 215.50925925925918]


## Sandsynlighedsfordeling

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
vals = np.linspace(0, 3, 100)
axes[0].plot(vals, graph.pdf(vals))
axes[1].plot(vals, graph.cdf(vals))
plt.show()

PDF beskriver sandsynligheden for coalescent tid.
CDF beskriver sandsynligheden for at MRCA er nået før tid t.

## Simulation

In [ ]:
samples = graph.sample(10000)
samples[:10]
np.mean(samples).item(), graph.expectation()

Sample mean: 1.4952193693448914
Theoretical mean: 1.5


Her sammenlignes den simulerede middelværdi med den teoretiske forventning.

In [ ]:
def mesh(state, max_val=None):
    transitions = []
    for i in range(state.size):
        if state[i] <= max_val:
            child = state.copy()
            child[i] += 1
            trans = [child, state.sum()]
            transitions.append(trans)
    return transitions

graph = Graph(mesh, ipv=[1, 1], max_val=2)
graph.plot()

In [ ]:
graph = Graph(mesh, ipv=[1, 1], max_val=8)
print(graph.expectation())
graph.plot()

## Konklusion

* Coalescent kan formuleres som en Markov jump process
* States repræsenterer lineage-struktur
* Transition rates følger klassisk coalescent teori
* Phasic gør det muligt at:
  * beregne eksakte fordelinger
  * simulere data
  * lave inference

Dette bliver fundamentet for:

* migration modeller
* IM modeller
* ghost populations



## Coalescent-modellen

I dette afsnit introduceres *Coalescent-modellen*, som er en klassisk Markov jump proces inden for populationsgenetik.

Modellen beskriver, hvordan en stikprøve af individer kan spores bagud i tid til en fælles forfader. I processen coalescerer linjer sammen, så antallet af slægtslinjer gradvist reduceres, indtil der kun er en fælles forfader tilbage.

Man følger ikke individer fremad i tid, men derimod deres *fælles ophav bagud i tid*.


### Tilstande i modellen

State i Coalescent-modellen kan beskrives som en vektor, der tæller hvor mange grupper (linjer) der har et bestemt antal efterkommere:

- 1-ton: linjer med en efterkommer (singleton)
- 2-ton: linjer med to efterkommere (doubleton)
- osv.

For eksempel kan en tilstand med 4 individer skrives som:

(4, 0, 0, 0)

Det betyder:

- 4 singleton-linjer
- 0 doubletons
- 0 tripletons
- 0 quadrupletons

Processen starter her og ender med:

(0, 0, 0, 1)

hvor alle individer deler en fælles forfader.

Når to linjer coalescerer:

- antallet af linjer falder med 1
- størrelsen på den nye gruppe øges

Over tid dannes et træ, hvor grenene repræsenterer slægtslinjer. Dette kaldes et *coalescent tree*.


### Hvorfor er modellen vigtig?

Coalescent-modellen bruges bl.a. til:

- genetisk dataanalyse
- forståelse af evolution
- estimering af populationsstørrelser
- modellering af mutationer

### Sammenhæng med Phasic

Coalescent-modellen kan implementeres i Phasic ved:

- at beskrive states som vektorer
- at definere en callback-funktion, der beskriver hvordan linjer coalescerer
- at angive rater for, hvor hurtigt coalescens sker

Dette gør det muligt at beregne:

- ventetider
- fordelinger
- forventede værdier

på samme måde som i de simple modeller tidligere – men nu for en langt mere kompleks proces.

In [ ]:
import seaborn as sns
from vscodenb import set_vscode_theme
set_vscode_theme()
sns.set_palette('tab10')

## Konstruktion af state space

I de tidligere eksempler blev modeller defineret direkte via en callback-funktion. 
I mere komplekse modeller som Coalescent kan det være nyttigt at konstruere hele state space eksplicit.

State space består af:

- alle mulige states processen kan være i
- alle overgange mellem disse state
- tilhørende rater

Man starter i en initial state, fx:
(4, 0, 0, 0)

Derefter:

1. Finder man alle mulige coalescent events
2. Opretter nye states
3. Tilføjer transitions mellem dem
4. Gentager indtil alle state er gennemgået


Det man ser er, at når to linjer coalescerer:

- en linje fjernes fra hver gruppe
- en ny gruppe med samlet størrelse tilføjes

Eksempel:
$$
(2,1,0,0) \rightarrow (1,0,1,0)
$$

Denne metode giver fuld kontrol over:

- hvilke state der findes
- hvordan rater beregnes


In [ ]:
import seaborn as sns
%config InlineBackend.figure_format = 'svg'
from vscodenb import set_vscode_theme

np.random.seed(42)
set_vscode_theme()
sns.set_palette('tab10')

In [ ]:
def coalescent_graph(nr_samples):
    state_vector_length = nr_samples
    graph = Graph(state_vector_length)

    initial_state = np.zeros(state_vector_length, dtype=int)
    initial_state[0] = nr_samples

    vertex = graph.find_or_create_vertex(initial_state)
    graph.starting_vertex().add_edge(vertex, 1)

    index = 1
    while index < graph.vertices_length():
        vertex = graph.vertex_at(index)
        state = vertex.state()

        for i in range(nr_samples):
            for j in range(i, nr_samples):
                same = int(i == j)
                if same and state[i] < 2:
                    continue
                if not same and (state[i] < 1 or state[j] < 1):
                    continue
                new_state = state.copy()
                new_state[i] -= 1
                new_state[j] -= 1
                new_state[i+j+1] += 1
                new_vertex = graph.find_or_create_vertex(new_state)
                rate = state[i]*(state[j]-same)/(1+same)
                vertex.add_edge(new_vertex, rate)
        index += 1
    return graph

In [ ]:
graph = coalescent_graph(4)
graph.plot()

In [ ]:
graph.states()

Her kan man se alle states i modellen. 
Hver række repræsenterer en mulig konfiguration af lineager.

In [ ]:
graph.plot()

## Sammenligning af metoder

Der findes to måder at opbygge modellen på:

1. state-space konstruktion
2. Callback-funktion

Callback-metoden er:

- kortere
- mere fleksibel
- lettere at vedligeholde

State-space konstruktion er:

- mere eksplicit
- nyttig til forståelse
- bedre hvis man vil manipulere grafen direkte



## State properties og indexing

Denne afsnit viser brugen af StateIndexer fra phasic til at repræsentere komplekse states i en coalescent-model.

Jeg fokuserer på:

- mapping mellem indeks og egenskaber
- kombinatoriske states
- anvendelse i en two-locus ARG

In [ ]:
from phasic import Graph, with_ipv, StateIndexer, PropertySet, Property
import numpy as np
import seaborn as sns
%config InlineBackend.figure_format = 'svg'
from vscodenb import set_vscode_theme
set_vscode_theme()

### Properties

En Property repræsenterer en dimension i state space.

In [ ]:
descendants = Property('descendants', max_value=10)
descendants

### Single PropertySet

In [ ]:
indexer = StateIndexer(
    lineage=[Property('descendants', max_value=10)]
)
print(f"Total states: {indexer.state_length}")
print(f"PropertySet: {indexer.lineage}")

In [ ]:
# Index to properties (returns dataclass by default)
props = indexer.lineage.i2p(5)
print(f"Index 5: {props}")
print(f"Access via attribute: props.descendants = {props.descendants}")

# Can also get as dict
props_dict = indexer.lineage.i2p(5, as_dict=True)
print(f"As dict: {props_dict}")

# Properties to index
idx = indexer.lineage.p2i({'descendants': 5})
print(f"{{descendants: 5}}: index {idx}")

# Also accepts dataclass
idx = indexer.lineage.p2i(props)
print(f"Dataclass: index {idx}")

### Combinatorial State Space with multiple properties

In [ ]:
# Two properties: derived allele status (0 or 1) × descendants (0 to 4)
indexer2 = StateIndexer(
    lineage=[
        Property('derived', max_value=1),      # 2 values: 0, 1
        Property('descendants', max_value=4)   # 5 values: 0, 1, 2, 3, 4
    ]
)

print(f"Total states: {indexer2.state_length} (2 × 5 = 10)")
print(f"\nAll states:")
for i in indexer2.lineage:
    props = indexer2.lineage.i2p(i)
    print(f"  Index {i:2d}: derived={props.derived}, descendants={props.descendants}")

### Two-locus ancestral recombination graph

Jeg modellerer lineager med antal efterkommere ved to loci.

In [ ]:
nr_samples = 4
indexer = StateIndexer(
    n_descendants=[
        Property('locus1', max_value=nr_samples),
        Property('locus2', max_value=nr_samples)
    ]
)
indexer

In [ ]:
indexer.state_length

In [ ]:
indexer.n_descendants

In [ ]:
print('index', 'locus1', 'locus2', sep='\t')

for i in indexer.n_descendants: 
    props = indexer.n_descendants.index_to_props(i) 
    print(i, props.locus1, props.locus2, sep='\t')

In [ ]:
props = indexer.n_descendants.index_to_props(5) 
props.locus1, props.locus2

In [ ]:
indexer.n_descendants.props_to_index(props)

In [ ]:
indexer.n_descendants.props_to_index(locus1=props.locus1 + 1, 
                                     locus2=props.locus2)

In [ ]:
# zero state vector with appropriate size
initial = [0] * indexer.state_length

# set initial state with all lineages having one descendant at both loci
initial[indexer.props_to_index(locus1=1, locus2=1)] = nr_samples

@with_ipv(initial)
def two_locus_arg(state, indexer=None, N=None, R=None):

    transitions = []
    if state.sum() <= 1: return transitions

    for i in range(indexer.state_length):
        if state[i] == 0: continue
        props_i = indexer.n_descendants.index_to_props(i)

        for j in range(i, indexer.state_length):
            if state[j] == 0: continue
            props_j = indexer.n_descendants.index_to_props(j)
            
            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue 
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            locus1 = props_i.locus1 + props_j.locus1
            locus2 = props_i.locus2 + props_j.locus2
            if locus1 <= nr_samples and locus2 <= nr_samples:
                child[indexer.props_to_index(locus1=locus1, locus2=locus2)] += 1
                transitions.append([child, state[i]*(state[j]-same)/(1+same)/N])

        if state[i] > 0 and props_i.locus1 > 0 and props_i.locus2 > 0:
            child = state.copy()
            child[i] -= 1
            child[indexer.props_to_index(locus1=props_i.locus1, locus2=0)] += 1
            child[indexer.props_to_index(locus1=0, locus2=props_i.locus2)] += 1
            transitions.append([child, R])

    return transitions


graph = Graph(two_locus_arg, 
                     N=1, R=1,
                     indexer=indexer) # passing indexer as argument
graph.plot(max_nodes=200)

In [ ]:
graph.expectation()

In [ ]:
reward_matrix = graph.states().T

In [ ]:
idx_singleton_loc1 = indexer.n_descendants.props_to_index(locus1=1)
idx_singleton_loc1

In [ ]:
reward_matrix = graph.states().T

singleton_rewards_loc1 = reward_matrix[idx_singleton_loc1].sum(axis=0)
singleton_rewards_loc1

In [ ]:
graph.expectation(singleton_rewards_loc1)

In [ ]:
sfs_loc1 = [graph.expectation(reward_matrix[indexer.n_descendants.props_to_index(locus1=i)].sum(axis=0)) for i in range(1, nr_samples+1)]
labels = [f"{i+1}'ton" for i in range(nr_samples)]
sns.barplot(x=labels, y=sfs_loc1, hue=labels) ; 